# NOAI Day 3 — scikit-learn

**Companion to NOAI deck Day 3 (4-topic core).** Scope: **Intro · Preprocessing · Pipeline · Regression**, plus Pitfalls. **12 topics, 2 questions each**:

- **Q1 — recall**: set the answer variable(s), run the check.
- **Q2 — applied**: write real sklearn code, run the `assert`.

No inline hints. **All solutions in the Solutions section at the bottom** — attempt first, then scroll.

Datasets are sklearn built-ins only — runs offline, no downloads.

Mantra: **fit on train · transform everywhere · Pipeline always.**

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/noai_day3_sklearn.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import (accuracy_score, r2_score,
                             mean_absolute_error, mean_squared_error)
np.random.seed(0)
print("imports ok")

---
## Topic 1 — What kind of ML problem is this?

A startup brings you three jobs. Before writing any code you must decide what
*type* of ML problem each one is — that choice decides which models you can use.

**Q1 (recall) — name the problem type.**
For each job, decide if it is **classification** (predict a category),
**regression** (predict a number), or **unsupervised** (no labels, find
structure). Put the three answers, in order, into the list `kinds`:

- (a) Predict the monthly rent of an apartment, in dollars.
- (b) Decide whether an incoming email is spam — yes or no.
- (c) You have shopper purchase histories but **no labels**; group similar shoppers.

**Q2 (applied) — train your first classifier.**
Use the built-in **Iris** dataset (predict the flower species from 4 measurements).

1. Load it into `X` (features) and `y` (species label).
2. Split 80% train / 20% test: `train_test_split(..., test_size=0.2,
   random_state=42, stratify=y)`.
3. Fit `LogisticRegression(max_iter=1000)` on the training set.
4. Set `test_acc` = the model's accuracy on the **test** set (use `.score`).

Target: `test_acc > 0.9`.

> 🧠 **Concept / Mindset.** Naming the problem type comes *before* any
> model — it decides the whole toolbox. **Mindset:** never reach for an
> algorithm until you can say "classification / regression / unsupervised"
> out loud.

In [ ]:
# TODO
kinds = [..., ..., ...]   # [(a), (b), (c)]

In [ ]:
assert kinds == ["regression", "classification", "unsupervised"]
print("T1 Q1 ok")

In [ ]:
# TODO
X, y = ...
X_tr, X_te, y_tr, y_te = ...
model = ...
test_acc = ...
print("acc:", round(test_acc, 3))

In [ ]:
assert test_acc > 0.9
print("T1 Q2 ok")

---
## Topic 2 — Splitting data: `random_state` & `stratify`

You hand a teammate your train/test split. They ask: "If I rerun this, do I get
the *same* split?" and "Our classes are unbalanced — how do I keep the same
class ratio in train and test?"

**Q1 (recall) — two facts about `train_test_split`.**
- `reproducible` → `True`/`False`: does a fixed `random_state` make the split
  return the **same rows every run**?
- `stratify_when` → `"classification"` or `"regression"`: for which problem
  type do we pass `stratify=y` (to keep class proportions)?

**Q2 (applied) — a stratified split on imbalanced data.**
You get 200 samples, 4 features each. The label is **imbalanced**: 160 of
class 0 and only 40 of class 1 (20% positives).

1. Split with `test_size=0.25`, `random_state=42`, `stratify=y`.
2. Set `n_test` = number of rows in the test set.
3. Set `ratio` = fraction of the test set that is class 1.

Because the split is stratified, `ratio` stays ≈ 0.20 (the original rate).

> 🧠 **Concept / Mindset.** The split simulates the future you don't
> have yet. **Mindset:** fix `random_state` so a result is reproducible,
> `stratify` so rare classes don't vanish — the test set must look like
> reality, not luck.

In [ ]:
# TODO
reproducible = ...        # True / False
stratify_when = ...       # "classification" or "regression"

In [ ]:
assert reproducible is True
assert stratify_when == "classification"
print("T2 Q1 ok")

In [ ]:
X = np.arange(800).reshape(200, 4)
y = np.array([0] * 160 + [1] * 40)
# TODO
X_tr, X_te, y_tr, y_te = ...
n_test = ...
ratio  = ...
print(n_test, round(ratio, 3))

In [ ]:
assert n_test == 50
assert abs(ratio - 0.2) < 1e-9
print("T2 Q2 ok")

---
## Topic 3 — The shapes sklearn expects: 2-D `X`, 1-D `y`

Classic beginner error: pass a flat Python list as features and get
`Reshape your data...`. sklearn is strict about shapes.

**Q1 (recall) — required dimensions.**
- `X_ndim` → how many dimensions the **feature matrix** `X` must have.
- `y_ndim` → how many dimensions the **target** `y` must have.

(In words: features are a table of rows × columns; the target is one column
of answers.)

**Q2 (applied) — shape it, split it, predict held-out.**
One feature `raw = [1, 2, 3, 4, 5, 6]` and true outputs
`y = [2, 4, 6, 8, 10, 12]` (so `y = 2 × raw`).

1. Turn `raw` into a **2-D** array `X2` of shape `(6, 1)` — 6 rows, 1 feature.
2. Split into train/test (`test_size=2`, `random_state=42`).
3. Fit `LinearRegression` model `m` **on the training rows only**.
4. `test_pred` = predictions on the held-out `X_te`.
5. `pred7` = prediction for an unseen input of `7` (pass it as `[[7]]`).

Expected: `X2.shape == (6, 1)`, `test_pred` matches `y_te` exactly (perfect
line), `pred7 ≈ 14`.

> 🧠 **Concept / Mindset.** sklearn wants a 2-D feature table and a 1-D
> target — shape is a contract, not a suggestion. **Mindset:** a model is
> only trusted on data it never trained on, so even a "shape" drill ends
> with a prediction on held-out rows, never on the training data.

In [ ]:
# TODO
X_ndim = ...   # X must be ?-D
y_ndim = ...   # y must be ?-D

In [ ]:
assert X_ndim == 2 and y_ndim == 1
print("T3 Q1 ok")

In [ ]:
raw = [1, 2, 3, 4, 5, 6]
y = np.array([2, 4, 6, 8, 10, 12])      # y = 2 * raw
# TODO
X2 = ...                                 # 2-D, shape (6, 1)
X_tr, X_te, y_tr, y_te = ...             # split: test_size=2, random_state=42
m = ...                                  # LinearRegression, fit on TRAIN only
test_pred = ...                          # predict on held-out X_te
pred7 = ...                              # predict for unseen 7  -> expect 14
print(X2.shape, np.round(test_pred, 3), round(float(pred7), 3))

In [ ]:
assert X2.shape == (6, 1)
assert np.allclose(test_pred, y_te)      # perfect line -> exact on held-out
assert abs(float(pred7) - 14) < 1e-6
print("T3 Q2 ok")

---
## Topic 4 — `StandardScaler`: fit on train only

Scaling rescales each feature to mean 0, std 1. Golden rule: the scaler
**learns its statistics from training data only**, then applies them to test
data — otherwise test info leaks into training.

**Q1 (recall) — which models care about scale?**
- `knn_needs_scaling` → `True`/`False`: does **k-NN** (distance-based) need scaling?
- `tree_needs_scaling` → `True`/`False`: does a **decision tree** need scaling?

**Q2 (applied) — scale by hand, leak-proof.**
One feature: `[10, 20, 30, 40, 50, 60]`. Train = first 4 rows, test = last 2.

1. Fit a `StandardScaler` **on the training rows only**.
2. Transform both train and test with that fitted scaler.
3. Set `mean_learned` = the mean the scaler learned (`scaler.mean_[0]`).
4. Set `tr_mean` = the mean of the **scaled training** data.

Expected: `mean_learned == 25.0` (mean of 10,20,30,40) and `tr_mean ≈ 0`
(scaled training data is always centred on 0).

> 🧠 **Concept / Mindset.** A scaler *learns* from train and *applies*
> everywhere. **Mindset:** the test set is the future — it must never
> influence what the scaler learns, or your score is a lie.

In [ ]:
# TODO
knn_needs_scaling  = ...   # True / False
tree_needs_scaling = ...   # True / False

In [ ]:
assert knn_needs_scaling is True
assert tree_needs_scaling is False
print("T4 Q1 ok")

In [ ]:
X = np.array([[10.], [20.], [30.], [40.], [50.], [60.]])
X_tr, X_te = X[:4], X[4:]
# TODO
sc = ...
X_tr_s = ...
X_te_s = ...
mean_learned = ...
tr_mean = ...
print(mean_learned, round(tr_mean, 9))

In [ ]:
assert abs(mean_learned - 25.0) < 1e-9
assert abs(tr_mean) < 1e-9
print("T4 Q2 ok")

---
## Topic 5 — Encoding categories: OneHot vs Ordinal

Models need numbers, not text. *How* you turn text into numbers depends on
whether the categories have a natural order.

**Q1 (recall) — pick the right encoder.**
- `encoder_for_city` → `"OneHotEncoder"` or `"OrdinalEncoder"`:
  a `city` column (Bangkok, Phuket, Chiang Mai) — **no order** between cities.
- `encoder_for_size` → a `size` column (S < M < L) — **has an order**.

**Q2 (applied) — one-hot encode a city column.**
DataFrame column `city = ["Bangkok", "Phuket", "Chiang Mai", "Bangkok"]`
— 3 distinct cities, 4 rows.

1. Fit `OneHotEncoder(sparse_output=False)` on the column, transform it.
2. Set `n_cols` = number of columns in the encoded output.
3. Set `total` = sum of every value in the encoded array.

Expected: `n_cols == 3` (one column per distinct city) and `total == 4`
(each of the 4 rows has exactly one 1).

> 🧠 **Concept / Mindset.** Numbers you invent carry meaning the model
> will believe. **Mindset:** order exists → Ordinal; no order → OneHot.
> Faking `0/1/2` on unordered cities tells the model Phuket > Bangkok.

In [ ]:
# TODO
encoder_for_city = ...   # cities have no order
encoder_for_size = ...   # S < M < L has order

In [ ]:
assert encoder_for_city == "OneHotEncoder"
assert encoder_for_size == "OrdinalEncoder"
print("T5 Q1 ok")

In [ ]:
df = pd.DataFrame({"city": ["Bangkok", "Phuket", "Chiang Mai", "Bangkok"]})
# TODO
ohe = ...
enc = ...
n_cols = ...
total = ...
print(n_cols, total)

In [ ]:
assert n_cols == 3
assert total == 4
print("T5 Q2 ok")

---
## Topic 6 — `SimpleImputer`: filling missing values

Real data has holes. `SimpleImputer` fills them with a learned statistic. The
right statistic depends on the column.

**Q1 (recall) — choose a strategy.**
- `strategy_outliers` → `"mean"`/`"median"`/`"most_frequent"`: a **numeric**
  column that has outliers (median resists outliers).
- `strategy_text` → for a **categorical / text** column.

**Q2 (applied) — fill numeric gaps with the median.**
Column `data = [25, 30, NaN, 45, NaN]` (two values missing).

1. Fit `SimpleImputer(strategy="median")`, transform `data`.
2. Set `med` = the statistic the imputer learned (median of 25, 30, 45).
3. Set `has_nan` = `True`/`False`: any NaN left after imputing?

Expected: `med == 30.0` and `has_nan is False`.

> 🧠 **Concept / Mindset.** Imputing replaces a hole with a *learned
> guess*. **Mindset:** pick the statistic that survives bad data — median
> for numbers with outliers, most_frequent for categories — and learn it
> from train only.

In [ ]:
# TODO
strategy_outliers = ...   # "mean" / "median" / "most_frequent"
strategy_text     = ...

In [ ]:
assert strategy_outliers == "median"
assert strategy_text == "most_frequent"
print("T6 Q1 ok")

In [ ]:
data = np.array([[25.], [30.], [np.nan], [45.], [np.nan]])
# TODO
imp = ...
filled = ...
med = ...
has_nan = ...
print(med, has_nan)

In [ ]:
assert med == 30.0
assert has_nan is False
print("T6 Q2 ok")

---
## Topic 7 — `fit` / `transform` and the leakage rule

`fit` *learns* parameters; `transform` *applies* them. Mixing these up across
train/test is the #1 cause of data leakage.

**Q1 (recall) — who calls what.**
- `test_method` → `"fit"`/`"transform"`/`"fit_transform"`: training data uses
  `fit_transform`; what should **test** data use?
- `leak_if_fit_all` → `True`/`False`: if you fit a scaler on the **full X
  before** splitting, is that data leakage?

**Q2 (applied) — show the leak-proof statistic.**
Feature `[1, 2, 3, 4, 5, 100]` (note the outlier `100`). Train = first 4 rows,
test = last 2.

1. Fit a `StandardScaler` **on train only**.
2. Transform the test rows with it.
3. Set `fit_mean` = `scaler.mean_[0]` (mean of 1,2,3,4 = 2.5).

The point: the huge `100` in the test set must **not** affect `fit_mean`.
Expected: `fit_mean ≈ 2.5`.

> 🧠 **Concept / Mindset.** `fit` learns, `transform` applies.
> **Mindset:** the moment test data touches `fit`, you've leaked the
> future into the past — train fits, everything else only transforms.

In [ ]:
# TODO
test_method = ...        # "fit" / "transform" / "fit_transform"
leak_if_fit_all = ...    # True / False

In [ ]:
assert test_method == "transform"
assert leak_if_fit_all is True
print("T7 Q1 ok")

In [ ]:
X = np.array([[1.], [2.], [3.], [4.], [5.], [100.]])
X_tr, X_te = X[:4], X[4:]
# TODO
sc = ...
X_te_s = ...
fit_mean = ...
print(fit_mean)

In [ ]:
assert abs(fit_mean - 2.5) < 1e-9
print("T7 Q2 ok")

---
## Topic 8 — `Pipeline`: chain steps, no leakage

A `Pipeline` bundles preprocessing + model into one object. `.fit` runs every
step in order on train; `.score` applies them to test — leakage becomes
impossible.

**Q1 (recall) — naming steps.**
Each step is `(name, estimator)`.
- `any_name_ok` → `True`/`False`: can you name a step almost anything?
- `dunder_allowed` → `True`/`False`: is a double underscore `__` allowed inside
  a step name? (It is reserved for addressing nested parameters.)

**Q2 (applied) — a full classification pipeline.**
Built-in **breast-cancer** dataset (predict benign vs malignant). The split is
already done for you (`random_state=42`, `stratify=y`).

Build a `Pipeline`, steps in order:
1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`
3. `LogisticRegression(max_iter=5000)`

Fit on train, set `acc` = test accuracy. Target: `acc > 0.95`.

> 🧠 **Concept / Mindset.** A `Pipeline` makes leakage *structurally*
> impossible. **Mindset:** if preprocessing lives outside the model
> object, someday someone fits it on everything — bundle it so they
> can't.

In [ ]:
# TODO
any_name_ok    = ...   # True / False
dunder_allowed = ...   # True / False  ("__" reserved for params)

In [ ]:
assert any_name_ok is True
assert dunder_allowed is False
print("T8 Q1 ok")

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
# TODO
pipe = ...
acc = ...
print(round(acc, 3))

In [ ]:
assert acc > 0.95
print("T8 Q2 ok")

---
## Topic 9 — `ColumnTransformer`: different prep per column

Numeric and categorical columns need different preprocessing.
`ColumnTransformer` applies a different transformer to each group of columns.

**Q1 (recall) — entry shape.**
A `ColumnTransformer` entry is `(name, transformer, columns)`.
Set `ct_tuple_len` = how many items in that tuple.

**Q2 (applied) — mixed numeric + categorical frame.**
DataFrame with numeric `age` (one NaN) and categorical `city` (one `None`):

```
age:  [25, 30, NaN, 40]
city: ["BKK", "HKT", "BKK", None]
```

Build a `ColumnTransformer` with two branches:
- numeric (`age`): `SimpleImputer(median)` → `StandardScaler`
- categorical (`city`): `SimpleImputer(most_frequent)` → `OneHotEncoder`

`fit_transform` the frame. Set `Xt` = transformed array, `n_out` = its column
count. Expected: 4 rows, `n_out == 3` (1 scaled numeric + 2 one-hot city
columns: BKK, HKT).

> 🧠 **Concept / Mindset.** Real tables are mixed-type. **Mindset:**
> numeric and categorical need different prep — *route* columns to the
> right transformer, never force one on all.

In [ ]:
# TODO
ct_tuple_len = ...

In [ ]:
assert ct_tuple_len == 3
print("T9 Q1 ok")

In [ ]:
df = pd.DataFrame({"age": [25., 30., np.nan, 40.],
                   "city": ["BKK", "HKT", "BKK", None]})
# TODO
num = ...
cat = ...
pre = ...
Xt = ...
n_out = ...
print(Xt.shape, n_out)

In [ ]:
assert Xt.shape[0] == 4
assert n_out == 3          # 1 scaled numeric + 2 city dummies
print("T9 Q2 ok")

---
## Topic 10 — `LinearRegression`: reading the fitted line

After fitting, sklearn stores everything it *learned* in attributes ending
with `_` (e.g. `coef_`). This separates learned values from settings you chose.

**Q1 (recall) — attribute names.**
- `slope_attr` → the attribute holding the slope/coefficients (a string).
- `intercept_attr` → the attribute holding the intercept (a string).

**Q2 (applied) — hours studied → exam score.**
You collected 1–5 hours of study (`X = [1,2,3,4,5]`) and scores
`y = [35, 42, 50, 55, 60]`.

1. Fit a `LinearRegression` model `m`.
2. Set `slope` = `m.coef_[0]` (points gained per extra hour).
3. Set `intercept` = `m.intercept_` (score at 0 hours).
4. Set `p6` = predicted score for **6** hours of study.

Expected roughly: `slope ≈ 6.3`, `intercept ≈ 29.5`.

> 🧠 **Concept / Mindset.** A trailing `_` means "learned from data",
> not "chosen by you". **Mindset:** read `coef_`/`intercept_` to see what
> the model actually learned — a model you can't interpret you can't
> trust.

In [ ]:
# TODO
slope_attr     = ...   # attribute name (string)
intercept_attr = ...

In [ ]:
assert slope_attr == "coef_"
assert intercept_attr == "intercept_"
print("T10 Q1 ok")

In [ ]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([35, 42, 50, 55, 60])
# TODO
m = ...
slope = ...
intercept = ...
p6 = ...
print(round(slope, 2), round(intercept, 2), round(p6, 2))

In [ ]:
assert abs(slope - 6.3) < 0.2
assert abs(intercept - 29.5) < 1.0
print("T10 Q2 ok")

---
## Topic 11 — Regression metrics (don't use accuracy!)

Accuracy is for classification. For regression you report **R²**, **MAE**,
**RMSE**. A regressor's `.score()` returns R² by default.

**Q1 (recall) — match metric to task.**
- `reg_score_returns` → `"r2"`/`"accuracy"`/`"mae"`: what does a regressor's
  `.score()` return?
- `accuracy_is_for` → `"classification"` or `"regression"`.

**Q2 (applied) — diabetes regression, reported correctly.**
Built-in **diabetes** dataset (predict disease progression — a number).
Split is `random_state=42`.

1. Build `Pipeline(StandardScaler → LinearRegression)`, fit on train.
2. Predict the test set into `y_pred`.
3. Set `r2`, `mae`, `rmse` from the matching metric functions
   (`rmse` = square root of mean squared error).
4. Set `score` = `pipe.score(X_te, y_te)`.

Sanity check: `score` must equal `r2` (they are the same thing). Need
`r2 > 0.4`.

> 🧠 **Concept / Mindset.** The metric *defines* success. **Mindset:**
> accuracy on regression is nonsense — report R²/MAE/RMSE, and know that
> a regressor's `.score()` already returns R².

In [ ]:
# TODO
reg_score_returns = ...   # "r2" / "accuracy" / "mae"
accuracy_is_for   = ...   # "classification" / "regression"

In [ ]:
assert reg_score_returns == "r2"
assert accuracy_is_for == "classification"
print("T11 Q1 ok")

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42)
# TODO
pipe = ...
y_pred = ...
r2 = ...
mae = ...
rmse = ...
score = ...
print(round(r2, 3), round(mae, 1), round(rmse, 1))

In [ ]:
assert abs(score - r2) < 1e-9
assert r2 > 0.4
print("T11 Q2 ok")

---
## Topic 12 — Pitfalls: spot the mistake & ship it right

Final boss: combine everything and avoid every trap — wrong method on test,
wrong metric, wrong shape, leakage.

**Q1 (recall) — three rules in one.**
- `test_step` → `"fit"`/`"transform"`/`"fit_transform"`: what you call on test
  data with an **already-fitted** transformer.
- `reg_metric` → a valid **regression** metric: `"r2"`, `"mae"`, or `"rmse"`.
- `x_ndim` → required number of dimensions for `X`.

**Q2 (applied) — leak-proof mixed-type classifier.**
DataFrame: numeric `a` (has a NaN) + categorical `c`, with binary label `y`.

Build **one** `Pipeline` that does everything (so leakage is impossible):
- `ColumnTransformer`: numeric → impute+scale; categorical → impute+OneHot
- final step → `LogisticRegression`

Split with `stratify=y`, `random_state=42`, fit, set `acc`. Any value in
`[0, 1]` passes — the dataset is tiny; the graded part is the correct
structure.

> 🧠 **Concept / Mindset.** Every earlier trap in one place.
> **Mindset:** leak-proof `Pipeline` + right metric + right shape is the
> *default*, not the polish — build it correct first, optimize later.

In [ ]:
# TODO
test_step  = ...   # "fit" / "transform" / "fit_transform"
reg_metric = ...   # "r2" / "mae" / "rmse"
x_ndim     = ...

In [ ]:
assert test_step == "transform"
assert reg_metric in {"r2", "mae", "rmse"}
assert x_ndim == 2
print("T12 Q1 ok")

In [ ]:
df = pd.DataFrame({"a": [1., 2., np.nan, 4., 5., 6., 7., 8.],
                   "c": ["x", "y", "x", "y", "x", "y", "x", "y"]})
y = np.array([0, 1, 0, 1, 0, 1, 0, 1])
# TODO
num = ...
cat = ...
pre = ...
pipe = ...
X_tr, X_te, y_tr, y_te = ...
acc = ...
print(round(acc, 3))

In [ ]:
assert 0.0 <= acc <= 1.0
print("T12 Q2 ok")

---
# ✅ Solutions

Attempt every Topic above before reading. Each block: **Q1** answer, then **Q2** code.

### Topic 1 — sklearn API & problem typing

**Q1.**

```python
kinds = ["regression", "classification", "unsupervised"]
```
Continuous → regression; discrete category → classification; no labels → unsupervised.

**Q2.**

```python
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
test_acc = model.score(X_te, y_te)
```
`fit` learns, `score` returns accuracy for a classifier.

### Topic 2 — train_test_split: stratify & random_state

**Q1.**

```python
reproducible = True
stratify_when = "classification"
```
Same seed → same split. `stratify` preserves class ratio — classification only.

**Q2.**

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
n_test = len(X_te)
ratio  = y_te.mean()
```
`stratify=y` keeps the 80/20 class balance in the test split exactly.

### Topic 3 — X / y shapes

**Q1.**

```python
X_ndim = 2
y_ndim = 1
```
`X` is a 2D matrix (rows × features); `y` is a 1D vector.

**Q2.**

```python
X2 = np.array(raw).reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X2, y, test_size=2, random_state=42)
m = LinearRegression().fit(X_tr, y_tr)
test_pred = m.predict(X_te)
pred7 = m.predict([[7]])[0]
```
`reshape(-1, 1)` makes a single-feature column; the model is fit on **train
only**, then judged on held-out rows — the habit every model needs.

### Topic 4 — StandardScaler (fit on train only)

**Q1.**

```python
knn_needs_scaling = True
tree_needs_scaling = False
```
Distance / gradient models (KNN, SVM, NN, linear) want scaling. Trees don't.

**Q2.**

```python
sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr)
X_te_s = sc.transform(X_te)
mean_learned = sc.mean_[0]      # = mean of train [10,20,30,40] = 25
tr_mean = X_tr_s.mean()
```
Scaler learns mean/std from **train**; test is only transformed.

### Topic 5 — OneHot vs Ordinal

**Q1.**

```python
encoder_for_city = "OneHotEncoder"
encoder_for_size = "OrdinalEncoder"
```
No inherent order → OneHot. Real order → Ordinal.

**Q2.**

```python
ohe = OneHotEncoder(sparse_output=False).fit(df)
enc = ohe.transform(df)
n_cols = enc.shape[1]            # 3 distinct cities
total = enc.sum()                # one 1 per row → 4
```
Each row becomes exactly one hot column, so the matrix sums to #rows.

### Topic 6 — SimpleImputer

**Q1.**

```python
strategy_outliers = "median"
strategy_text = "most_frequent"
```
Median resists outliers; categorical columns use the most frequent value.

**Q2.**

```python
imp = SimpleImputer(strategy="median").fit(data)
filled = imp.transform(data)
med = imp.statistics_[0]              # median of [25,30,45] = 30
has_nan = bool(np.isnan(filled).any())
```
`statistics_` holds the value learned per column on `fit`.

### Topic 7 — fit / transform & the leakage rule

**Q1.**

```python
test_method = "transform"
leak_if_fit_all = True
```
Fitting on all data lets test statistics leak in → optimistic, fake scores.

**Q2.**

```python
sc = StandardScaler().fit(X_tr)
X_te_s = sc.transform(X_te)
fit_mean = sc.mean_[0]      # mean of train [1,2,3,4] = 2.5 (the 100 never seen)
```
The outlier 100 is in test, so it can't move the learned mean.

### Topic 8 — Pipeline

**Q1.**

```python
any_name_ok = True
dunder_allowed = False
```
Any unique name works; `__` is reserved (e.g. `model__C` in GridSearchCV).

**Q2.**

```python
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression(max_iter=5000)),
]).fit(X_tr, y_tr)
acc = pipe.score(X_te, y_te)
```
One `fit` chains every step; test is auto-`transform`-ed → no leakage.

### Topic 9 — ColumnTransformer

**Q1.**

```python
ct_tuple_len = 3
```
`(name, transformer, column_list)` — the bridge between the two prep pipes.

**Q2.**

```python
num = Pipeline([("i", SimpleImputer(strategy="median")),
                ("s", StandardScaler())])
cat = Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                ("o", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("num", num, ["age"]),
                         ("cat", cat, ["city"])])
Xt = pre.fit_transform(df)
n_out = Xt.shape[1]
```
2 distinct cities → 2 dummy columns, +1 numeric = 3.

### Topic 10 — LinearRegression

**Q1.**

```python
slope_attr = "coef_"
intercept_attr = "intercept_"
```
Trailing `_` = "learned on fit" (inspection philosophy).

**Q2.**

```python
m = LinearRegression().fit(X, y)
slope = m.coef_[0]
intercept = m.intercept_
p6 = m.predict([[6]])[0]
```
Best-fit line ≈ `score = 6.3*hours + 29.5`.

### Topic 11 — Regression metrics

**Q1.**

```python
reg_score_returns = "r2"
accuracy_is_for = "classification"
```
A regressor's `.score()` is R², not accuracy — and R² can go negative.

**Q2.**

```python
pipe = Pipeline([("s", StandardScaler()),
                 ("m", LinearRegression())]).fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
rmse = mean_squared_error(y_te, y_pred) ** 0.5
score = pipe.score(X_te, y_te)
```
`.score` on a regression pipeline == `r2_score(y_te, y_pred)`.

### Topic 12 — Pitfalls: spot & fix

**Q1.**

```python
test_step = "transform"
reg_metric = "r2"
x_ndim = 2
```
The three most common interview-killers: refit on test, accuracy for regression, 1D X.

**Q2.**

```python
num = Pipeline([("i", SimpleImputer(strategy="median")),
                ("s", StandardScaler())])
cat = Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                ("o", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("n", num, ["a"]), ("c", cat, ["c"])])
pipe = Pipeline([("pre", pre),
                 ("m", LogisticRegression(max_iter=1000))])
X_tr, X_te, y_tr, y_te = train_test_split(
    df, y, test_size=0.25, random_state=42, stratify=y)
pipe.fit(X_tr, y_tr)
acc = pipe.score(X_te, y_te)
```
Pipeline + ColumnTransformer = zero leakage, one deployable object.